In [ ]:
from utils.database_utils import generate_database_and_retriever
from neo4j import GraphDatabase
import json

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")


def get_graph_context(graph_driver, retrieved_nodes):
    """
    Takes the output from your MultiVectorRetriever and
    fetches a 2-hop neighborhood for each node.
    """

    all_knowledge_blocks = []
    with graph_driver.session() as session:
        for full_node_params in retrieved_nodes:
            params = {
                "name": full_node_params["name"],
                "label": full_node_params["type"],
                "description": full_node_params["description"],
            }
            # Assuming retrieved_nodes are Document objects from your retriever
            # We pull the name or ID from the metadata

            query = """
            MATCH (startNode)
            WHERE startNode.name = $name 
            AND $label IN labels(startNode) 
            AND startNode.description = $description

            MATCH path = (startNode)-[r*1..2]-(neighbor)
            WHERE startNode <> neighbor
            AND NONE(lbl IN labels(neighbor) WHERE lbl IN ['text', 'image', 'table', 'document'])
            AND NONE(rel IN relationships(path) WHERE type(rel) = 'BELONGS_TO')

            WITH startNode, neighbor, collect(path) AS paths
            RETURN 
                startNode.name AS source_name,
                labels(startNode)[0] AS source_label,
                neighbor.name AS target_name,
                labels(neighbor)[0] AS target_label,
                neighbor.description AS target_desc,
                /* Collect all directional paths leading to this specific neighbor */
                [p IN paths | [rel IN relationships(p) | {
                    subject: startNode(rel).name,
                    predicate: type(rel),
                    object: endNode(rel).name
                }]] AS connection_paths
            """

            def format_rel(rel_type):
                return rel_type.replace("_", " ").lower()

            results = session.run(query, **params)

            for record in results:
                path_narratives = []

                # Process each unique path found for this specific neighbor
                for path in record["connection_paths"]:
                    steps = [
                        f"'{step['subject']}' {format_rel(step['predicate'])} '{step['object']}'"
                        for step in path
                    ]
                    path_narratives.append(" -> ".join(steps))

                # Join multiple paths with a semicolon
                combined_paths = " ; also ".join(path_narratives)

                # Create a dense, summarized block
                summary_block = (
                    f"ENTITY PROFILE: {record['source_name']} ({record['source_label']})\n"
                    f"FACT: It is linked to the {record['target_label']} '{record['target_name']}' via these connections: {combined_paths}.\n"
                    f"CONTEXT FOR {record['target_name']}: {record['target_desc'] or 'No description.'}\n"
                )
                all_knowledge_blocks.append(summary_block)

            # Final context to pass to the LLM
            final_rag_context = "\n---\n".join(all_knowledge_blocks)

    final_rag_context = "\n---\n".join(all_knowledge_blocks)
    return final_rag_context


In [ ]:
from langchain_ollama import OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage


def summarize_graph_context(nodes_context):
    system_prompt = """

        You are an expert in knowledge synthesis and technical summarization.

        Your task is to transform structured knowledge about a model into a high-quality, dense, and retrieval-optimized summary for a RAG system.

        ### OBJECTIVE:
        Produce a concise but information-rich summary that:
        - Preserves all key technical details
        - Removes redundancy
        - Clearly separates concepts

        ### INPUT FORMAT:
        You will receive structured information including:
        - Relationships
        - Features
        - Targets
        - Metrics
        - Context

        ### OUTPUT REQUIREMENTS:

        1. Write a **dense paragraph summary** (5–8 sentences max)
        2. Then provide a **bullet-point structured summary** with:
        - Model role / purpose
        - Key relationships
        - Input features
        - Outputs
        - Evaluation metrics
        - Dataset details
        3. Normalize terminology (e.g., "sensitivity" = "recall" if appropriate)
        4. Remove duplicated or indirect relationships
        5. Prioritize direct, meaningful connections over graph paths
        6. Keep all numeric values and performance metrics
        7. Make the summary self-contained (no references to "the graph" or "above")

        ### STYLE:
        - Precise
        - Technical
        - Compact (high signal-to-noise)
        - No fluff or generic explanations

    """


In [13]:
def parse_nodes(nodes_retrieved):
    nodes_parsed = []
    for node in nodes_retrieved:
        nodes_parsed.append(json.loads(node.decode("utf-8")))
    return nodes_parsed

# Retrievers

You have to initiate 4 different retrievers: 
 - For regular rag
 - For nodes related to the search query
 - For mid level communities
 - For global level communities

In [3]:
regular_rag_retriever = retriever = generate_database_and_retriever(
    main_folder="./localdb"
)

nodes_retriever = generate_database_and_retriever(main_folder="./localdb/node_db")

mid_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/mid_communities"
)

global_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/global_communities"
)

In [4]:
docs_retrieved = retriever.invoke("liniar regression model")

nodes_retrieved = nodes_retriever.invoke("liniar regression model")
mid_level_communities_retrieved = mid_level_retriever.invoke("model")
global_level_communities_retrieved = global_level_retriever.invoke("model")


In [29]:
nodes_parsed = parse_nodes(nodes_retrieved)

In [47]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    nodes_context = get_graph_context(driver, nodes_parsed)

In [48]:
print(nodes_context)

ENTITY PROFILE: linear regression model (tech)
FACT: It is linked to the tech 'interpretable surrogate model' via these connections: 'linear regression model' is a 'interpretable surrogate model' ; also 'linear regression model' predicted by 'decision tree model' -> 'decision tree model' is a 'interpretable surrogate model'.
CONTEXT FOR interpretable surrogate model: interpretable surrogate model

---
ENTITY PROFILE: linear regression model (tech)
FACT: It is linked to the tech 'decision tree model' via these connections: 'linear regression model' is a 'interpretable surrogate model' -> 'decision tree model' is a 'interpretable surrogate model' ; also 'linear regression model' predicted by 'decision tree model' ; also 'roc curve' depicts 'linear regression model' -> 'roc curve' depicts 'decision tree model' ; also 'linear regression model' has attribute 'optimal shade' -> 'decision tree model' has attribute 'optimal shade'.
CONTEXT FOR decision tree model: A decision tree model is a su